<a href="https://colab.research.google.com/github/Jenn1217/huggingface-llm/blob/hfllm/chapter2/step3muti-sec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 处理多个序列 (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


# step1 准备工作，引入必要的库
1.   设定模型checkpoint 在这里指的是预训练模型的标识符或路径，可以理解为模型的"身份证"或"地址"。
2.  接着调用模型 **from_pretrained 是 Hugging Face Transformers 库中最重要的方法之一，用于加载预训练模型或分词器。**



In [93]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
#召唤模型
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)


# step2 调用分词器

In [94]:
tokenizer1 = AutoTokenizer.from_pretrained(checkpoint)

In [95]:
sequence = "I've been waiting for a HuggingFace course my whole life."

# 输出错误

In [102]:
tokens = tokenizer1.tokenize(sequence)
ids = tokenizer1.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)
input_ids2 = torch.tensor([ids])
print("torch.tensor(ids)的结果是：",input_ids)
print("torch.tensor([ids])的结果是：",input_ids2)

torch.tensor(ids)的结果是： tensor([ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
         2026,  2878,  2166,  1012])
torch.tensor([ids])的结果是： tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])


# Transformers 模型默认情况下需要一个句子列表 重点在这一段



**为什么要用 [ids] 而不是 ids？**

torch.tensor(ids)：形状为 [14]（一维）

torch.tensor([ids])：形状为 [1, 14]（二维，批次维度为1）

模型需要批次维度，即使只有一个样本

**input_ids**定义：
输入令牌的数值表示，每个单词/子词被映射到词汇表中的唯一ID

In [11]:

tokens = tokenizer1.tokenize(sequence)# 将文本分成令牌（tokens）
ids = tokenizer1.convert_tokens_to_ids(tokens)# 将令牌转换为对应的ID

# 3. 创建PyTorch张量并添加批次维度
# torch.tensor(ids)：将Python列表转换为PyTorch张量
# [ids]：添加批次维度（即使只有一个样本）
input_ids = torch.tensor([ids])
print("Input IDs:", input_ids)

#将输入传递给模型进行预测
output = model(input_ids)
print("Logits:", output.logits)

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


**output.logits**模型输出的原始分数（未归一化的概率）

output = model(input_ids)

print(output.logits)  # 示例输出：tensor([[ 2.1246, -1.8934]])

 logits解释：

 第一个值 (2.1246) → 类别0（如"负面"）的得分

 第二个值 (-1.8934) → 类别1（如"正面"）的得分


In [15]:
# 单句预测
output_single = model(input_ids)
print("单句 logits:", output_single.logits)

单句 logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


# batching 处理多个句子

批处理（Batching）是一次性通过模型发送多个句子的行为。如果你只有一句话，你可以构建一个只有一个句子的 batch：


In [16]:
batched_ids = [
    [ids],
    [ids]
]

将这个 batched_ids 列表转换为张量，并通过你的模型进行处理。

检查你是否得到了与之前相同的 logits 值（但是重复了两次）！

In [17]:
input_batch = torch.tensor(batched_ids)

In [29]:
batched_ids = [
    [200, 200, 200],
    [200, 200]
]

因为长度不同，不可以被转换成矩形张量

ValueError: expected sequence of length 3 at dim 1 (got 2)


In [30]:
batch32 = torch.tensor(batched_ids)

ValueError: expected sequence of length 3 at dim 1 (got 2)

In [25]:
padding_id = 100

batched_ids = [
    [200, 200, 200],
    [200, 200, padding_id],
]

In [26]:
batch33 = torch.tensor(batched_ids)

将这个 batched_ids **列表转换为张量**，并通过你的模型进行处理。

批处理预测中的 logits 值有点问题：第二行应该与第二句的 logits 相同，但我们得到了完全不同的值！

这是因为 Transformer 模型的关键特性：注意力层，它考虑了每个 token 的上下文信息。这具体来说，每个 token 的含义并非单独存在的，它的含义还取决于它在句子中的位置以及周围的其他 token，包括填充tokenid

In [32]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer1.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


# batching后的张量值与原来不同

In [ ]:
print(model(torch.tensor(batched_ids)).logits)

我们需要告诉这些注意层忽略填充 token。这是通过使用注意力掩码（attention mask）层来实现的。

## 注意力掩码（attention mask）层
注意力掩码（attention mask）是与 inputs ID 张量形状完全相同的张量，用 0 和 1 填充：

1 表示应关注相应的 tokens，0 表示应忽略相应的 tokens（即，它们应被模型的注意力层忽视）。

In [33]:
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer1.pad_token_id],
]

attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

In [34]:
outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


✏️ 试试看！在第二节使用的两个句子（“I’ve been waiting for a HuggingFace course my whole life.” 和 “I hate this so much!”）上手动进行 tokenize。将它们输入模型并检查你是否得到了与第二节相同的 logits 值。然后使用填充 token 将它们一起进行批处理，然后创建合适的注意力掩码。检查模型计算后是否得到了相同的结果！



In [46]:
str1="I’ve been waiting for a HuggingFace course my whole life."
str2="I hate this so much!"

#调用分词器
tokenizerstr1=AutoTokenizer.from_pretrained(checkpoint)
#转化为分词
str1word=tokenizerstr1.tokenize(str1)
print(str1word)
str2word=tokenizerstr1.tokenize(str2)
print(str2word)

#分词转化为ID
str1id=tokenizerstr1.convert_tokens_to_ids(str1word)
str2id=tokenizerstr1.convert_tokens_to_ids(str2word)
print(str1id)
print(str2id)

['i', '’', 've', 'been', 'waiting', 'for', 'a', 'hugging', '##face', 'course', 'my', 'whole', 'life', '.']
['i', 'hate', 'this', 'so', 'much', '!']
[1045, 1521, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]
[1045, 5223, 2023, 2061, 2172, 999]


In [52]:
pad_id = tokenizerstr1.pad_token_id
print(pad_id)

0


In [58]:
max_length = max(len(str1id), len(str2id))


In [59]:
str2id_padded = str2id + [pad_id] * (max_length - len(str2id))
print("填充后的str2id:", str2id_padded)

填充后的str2id: [1045, 5223, 2023, 2061, 2172, 999, 0, 0, 0, 0, 0, 0, 0, 0]


In [77]:
batching_strs00=[str1id,str2id_padded]
print(batching_strs)

[[1045, 1521, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012], [1045, 5223, 2023, 2061, 2172, 999, 0, 0, 0, 0, 0, 0, 0, 0]]


# 原始 str1\2 张量

In [89]:
str1idtensor=torch.tensor([str1id])
outputstr1 =model(str1idtensor)

print(outputstr1.logits)

tensor([[-2.5720,  2.6852]], grad_fn=<AddmmBackward0>)


In [90]:
str2idtensor=torch.tensor([str2id])
outputstr2 =model(str2idtensor)

print(outputstr2.logits)

tensor([[ 3.1931, -2.6685]], grad_fn=<AddmmBackward0>)


# 不干涉填充padding的结果

In [78]:
input_ids_tensor00=torch.tensor(batching_strs00)

In [81]:
outputstr00 = model(input_ids_tensor00)
print(outputstr00.logits)

tensor([[-2.5720,  2.6852],
        [ 2.5423, -2.1265]], grad_fn=<AddmmBackward0>)


# 错误：嵌套了多余的列表
# batching_strs = [[str1_padded], [str2_padded]]  # 形状会是 [2, 1, max_length]

注意，这里是三层的而不是两层的

In [92]:
batching_strs_wrong=[[str1id],[str2id_padded]]
print(batching_strs_wrong)

[[[1045, 1521, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]], [[1045, 5223, 2023, 2061, 2172, 999, 0, 0, 0, 0, 0, 0, 0, 0]]]


In [72]:
batching_strs=[str1id,str2id_padded]
print(batching_strs)

[[1045, 1521, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012], [1045, 5223, 2023, 2061, 2172, 999, 0, 0, 0, 0, 0, 0, 0, 0]]


# 创建注意力掩码（python 列表）
这个掩码是自己创建的，而不是原本有的

In [73]:
attention_mask12 = torch.tensor([
    [1] * len(str1id) + [0] * (max_length - len(str1id)),  # 句子1：真实token为1，填充为0
    [1] * len(str2id) + [0] * (max_length - len(str2id))   # 句子2：真实token为1，填充为0
])
print(f"注意力掩码: {attention_mask12.tolist()}")

注意力掩码: [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]


转换成为tensor，PyTorch神经网络模型只能处理张量，不能直接处理Python列表：

In [ ]:
input_ids_tensor = torch.tensor(batching_strs)  # 形状应该是 [2, max_length]
attention_mask_tensor = torch.tensor(attention_mask12)  # 形状应该是 [2, max_length]

print("输入张量形状:", input_ids_tensor.shape)
print("注意力掩码形状:", attention_mask_tensor.shape)

In [74]:
# 6. 模型预测（使用修正后的张量）
try:
    outputstr12 = model(input_ids_tensor, attention_mask=attention_mask_tensor)
    print("成功! Logits:", outputstr12.logits)
except Exception as e:
    print("错误:", e)

输入张量形状: torch.Size([2, 14])
注意力掩码形状: torch.Size([2, 14])
成功! Logits: tensor([[-2.5720,  2.6852],
        [ 3.1931, -2.6685]], grad_fn=<AddmmBackward0>)


/tmp/ipython-input-2810087610.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask_tensor = torch.tensor(attention_mask12)  # 形状应该是 [2, max_length]


In [91]:
 print(outputstr12.logits)

tensor([[-2.5720,  2.6852],
        [ 3.1931, -2.6685]], grad_fn=<AddmmBackward0>)
